In [0]:
# COMMAND ----------

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType,
    DateType
)

from datetime import datetime

print("Libraries imported successfully")

In [0]:
# COMMAND ----------

CATALOG = "retail_ai_prod"
GOLD_SCHEMA = "gold_data"

VOLUME_PATH = (
    "/Volumes/retail_ai_prod/"
    "unstructured_docs/retail_documents"
)

SOURCE_PATH = f"{VOLUME_PATH}/landing/structured_data"

SALES_PATH = f"{SOURCE_PATH}/sales_transactions.csv"
INVENTORY_PATH = f"{SOURCE_PATH}/inventory_levels.csv"
CUSTOMERS_PATH = f"{SOURCE_PATH}/customer_profiles.csv"

SALES_TABLE = f"{CATALOG}.{GOLD_SCHEMA}.sales_transactions"
INVENTORY_TABLE = f"{CATALOG}.{GOLD_SCHEMA}.inventory_levels"
CUSTOMER_TABLE = f"{CATALOG}.{GOLD_SCHEMA}.customer_profiles"

print("Source:", SOURCE_PATH)
print("Destination:", CATALOG, GOLD_SCHEMA)

In [0]:
# COMMAND ----------

sales_schema = StructType([
    StructField("transaction_id", StringType(), False),
    StructField("transaction_date", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("store_city", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("gross_sales", DoubleType(), True),
    StructField("discount_amount", DoubleType(), True),
    StructField("returns_amount", DoubleType(), True),
    StructField("net_revenue", DoubleType(), True),
    StructField("payment_method", StringType(), True),
    StructField("channel", StringType(), True)
])

print("Sales schema defined")

In [0]:
# COMMAND ----------

inventory_schema = StructType([
    StructField("snapshot_date", StringType(), True),
    StructField("product_id", StringType(), False),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("store_city", StringType(), True),
    StructField("on_hand_qty", IntegerType(), True),
    StructField("reserved_qty", IntegerType(), True),
    StructField("reorder_level", IntegerType(), True),
    StructField("unit_cost", DoubleType(), True),
    StructField("supplier_id", StringType(), True)
])

print("Inventory schema defined")

In [0]:
# COMMAND ----------

customer_schema = StructType([
    StructField("customer_id", StringType(), False),
    StructField("customer_name", StringType(), True),
    StructField("city", StringType(), True),
    StructField("signup_date", StringType(), True),
    StructField("loyalty_tier", StringType(), True),
    StructField("preferred_channel", StringType(), True),
    StructField("age_band", StringType(), True)
])

print("Customer schema defined")

#Reading

In [0]:
# COMMAND ----------

sales_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("delimiter", ",")
    .option("mode", "FAILFAST")
    .schema(sales_schema)
    .load(SALES_PATH)
)

display(sales_df)

In [0]:
# COMMAND ----------

inventory_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("mode", "FAILFAST")
    .schema(inventory_schema)
    .load(INVENTORY_PATH)
)

display(inventory_df)

In [0]:
# COMMAND ----------

customer_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("mode", "FAILFAST")
    .schema(customer_schema)
    .load(CUSTOMERS_PATH)
)

display(customer_df)

#Velidating

In [0]:
# COMMAND ----------

sales_df = sales_df.withColumn(
    "transaction_date",
    F.to_date("transaction_date", "yyyy-MM-dd")
)

inventory_df = inventory_df.withColumn(
    "snapshot_date",
    F.to_date("snapshot_date", "yyyy-MM-dd")
)

customer_df = customer_df.withColumn(
    "signup_date",
    F.to_date("signup_date", "yyyy-MM-dd")
)

print("Date conversion completed")

In [0]:
# COMMAND ----------

sales_count = sales_df.count()
inventory_count = inventory_df.count()
customer_count = customer_df.count()

print("Sales records:", sales_count)
print("Inventory records:", inventory_count)
print("Customer records:", customer_count)

##Writing into tables

In [0]:
# COMMAND ----------

(
    sales_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SALES_TABLE)
)

print("Sales Delta table created!")

In [0]:
# COMMAND ----------

(
    inventory_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(INVENTORY_TABLE)
)

print("Inventory Delta table created!")

In [0]:
# COMMAND ----------

(
    customer_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(CUSTOMER_TABLE)
)

print("Customer Delta table created!")

In [0]:
%sql
-- COMMAND ----------

COMMENT ON TABLE
retail_ai_prod.gold_data.sales_transactions
IS 'Curated retail sales transaction data containing gross sales, discounts, returns and net revenue';

COMMENT ON TABLE
retail_ai_prod.gold_data.inventory_levels
IS 'Curated inventory snapshots by product and store city, including available stock and reorder levels';

COMMENT ON TABLE
retail_ai_prod.gold_data.customer_profiles
IS 'Curated customer master data containing location, loyalty tier and channel preferences';

##Verification

In [0]:
%sql
-- COMMAND ----------

SELECT
    store_city,
    ROUND(SUM(net_revenue), 2) AS total_revenue
FROM retail_ai_prod.gold_data.sales_transactions
GROUP BY store_city
ORDER BY total_revenue DESC;

In [0]:
%sql
-- COMMAND ----------

SELECT
    product_id,
    product_name,
    SUM(quantity) AS total_quantity_sold,
    ROUND(SUM(net_revenue), 2) AS total_revenue
FROM retail_ai_prod.gold_data.sales_transactions
GROUP BY product_id, product_name
ORDER BY total_revenue DESC
LIMIT 10;

In [0]:
%sql
-- COMMAND ----------

SELECT
    product_id,
    product_name,
    store_city,
    on_hand_qty,
    reserved_qty,
    reorder_level
FROM retail_ai_prod.gold_data.inventory_levels
WHERE on_hand_qty - reserved_qty < reorder_level
ORDER BY store_city;

In [0]:
%sql
-- COMMAND ----------

SELECT
    loyalty_tier,
    COUNT(*) AS total_customers
FROM retail_ai_prod.gold_data.customer_profiles
GROUP BY loyalty_tier
ORDER BY total_customers DESC;

In [0]:
%sql
-- COMMAND ----------

SELECT
    DATE_FORMAT(transaction_date, 'yyyy-MM') AS sales_month,
    ROUND(SUM(gross_sales), 2) AS gross_sales,
    ROUND(SUM(returns_amount), 2) AS total_returns,
    ROUND(SUM(net_revenue), 2) AS net_revenue
FROM retail_ai_prod.gold_data.sales_transactions
GROUP BY DATE_FORMAT(transaction_date, 'yyyy-MM')
ORDER BY sales_month;